# UIT DSC 2026 – LegalQA Generation-Only Smoke Test

Notebook ngắn để kiểm tra **Vi-Qwen2-1.5B-RAG có rút gọn/cắt câu trả lời hay không** mà không xây lại BM25, Dense FAISS hoặc Reranker.

Quy trình:

1. Tự tìm `train.json`, public test và `selected-contexts` theo cùng cơ chế của notebook chính.
2. Ưu tiên `submission_v0.json` làm `CONTEXT` nếu tìm thấy; nếu không có thì dùng các đáp án train dài.
3. So sánh prompt chống tóm tắt ở `max_new_tokens=512` và `1024`.
4. Báo `length_ratio`, mức bảo toàn token, boilerplate, từ chối và dấu hiệu chạm giới hạn token.

> Trên Kaggle, hãy **Add Data** giống notebook chính (dataset LegalQA và output model snapshot nếu có). Nếu không có snapshot, bật Internet để tải model từ Hugging Face. Notebook này không đánh giá retrieval; nó chỉ trả lời câu hỏi “generation có đang rút gọn quá mức không?”.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# ===== CẤU HÌNH NHANH =====
REPO_URL = "https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git"
GENERATOR_MODEL_ID = "AITeamVN/Vi-Qwen2-1.5B-RAG"
TEST_LIMIT = 10                 # 10 ID x 2 cấu hình; tăng lên 20 nếu muốn
RUN_1024_TEST = True            # False nếu chỉ muốn test 512 nhanh hơn
USE_ACTUAL_QUESTIONS = True     # False: dùng câu lệnh kiểm tra chung
MAX_INPUT_TOKENS_512 = 7168
MAX_INPUT_TOKENS_1024 = 6144
REPETITION_PENALTY = 1.05
SEED = 2026

# Ghi đè đường dẫn nếu auto-discovery chọn sai; để None để tự tìm.
DATASET_DIR = None
TRAIN_PATH = None
PUBLIC_PATH = None
CONTEXTS_PATH = None
SOURCE_SUBMISSION_PATH = None       # Tùy chọn: dùng answer của submission làm CONTEXT
COMPARISON_SUBMISSION_PATH = None   # Tùy chọn: so sánh độ rút ngắn với submission khác
MODEL_PATH = None

# Các ID đã thấy bị rút mạnh/từ chối trong v1.
PRIORITY_IDS = ["34235", "62147", "86293", "80189", "135669"]

if Path("/kaggle/working").is_dir():
    PLATFORM = "Kaggle"
    REPO_DIR = Path("/kaggle/working/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/kaggle/input")
    WORK_DIR = Path("/kaggle/working/legalqa-run")
    EXPORT_DIR = Path("/kaggle/working")
elif Path("/content").is_dir():
    PLATFORM = "Colab"
    REPO_DIR = Path("/content/uit-dsc-2026-task2-legalqa")
    INPUT_ROOT = Path("/content")
    WORK_DIR = Path("/content/legalqa-run")
    EXPORT_DIR = WORK_DIR
else:
    PLATFORM = "Local"
    REPO_DIR = Path(".").resolve()
    if not (REPO_DIR / "legalqa_baseline").is_dir():
        candidate = REPO_DIR / "UIT_DSC_2026_LegalQA_baseline_v0.1"
        if candidate.is_dir():
            REPO_DIR = candidate
    INPUT_ROOT = REPO_DIR
    WORK_DIR = REPO_DIR / "artifacts"
    EXPORT_DIR = WORK_DIR

HF_CACHE_DIR = WORK_DIR / "hf-cache"
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")

print(f"Platform: {PLATFORM}")
print(f"Input root: {INPUT_ROOT}")
print(f"Repo: {REPO_DIR}")
print(f"Work dir: {WORK_DIR}")
print(f"Output: {EXPORT_DIR}")


## 1. Cài thư viện tối thiểu

Chỉ cài generator dependencies; không cài FAISS, embedding hoặc reranker.


In [ ]:
required = {
    "transformers": "transformers>=4.40.0",
    "accelerate": "accelerate",
    "sentencepiece": "sentencepiece",
    "pandas": "pandas",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Dependencies already available.")

if not (REPO_DIR / "legalqa_baseline").is_dir():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(f"REPO_DIR tồn tại nhưng không phải project LegalQA: {REPO_DIR}")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import pandas as pd
import torch
from IPython.display import FileLink, Markdown, display
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from legalqa_baseline.generator import ViQwenRAGGenerator

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    for gpu_id in range(torch.cuda.device_count()):
        print(f"  GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)}")


## 2. Tự tìm dataset, submission tùy chọn và model snapshot


In [ ]:
MODEL_MARKER = ".legalqa_model.json"


def search_roots():
    candidates = [
        Path(DATASET_DIR) if DATASET_DIR is not None else None,
        INPUT_ROOT,
        WORK_DIR,
        REPO_DIR,
        REPO_DIR.parent,
        Path("./data"),
        Path(".").resolve(),
    ]
    seen = set()
    for root in candidates:
        if root is None:
            continue
        root = Path(root)
        if not root.exists():
            continue
        key = str(root.resolve())
        if key not in seen:
            seen.add(key)
            yield root


def resolve_file(label, override, accepted_names, required=True):
    if override is not None:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f"{label} không tồn tại: {path}")
        return path

    accepted = {name.casefold() for name in accepted_names}
    for root in search_roots():
        matches = sorted(
            (path for path in root.rglob("*") if path.is_file() and path.name.casefold() in accepted),
            key=lambda path: (len(path.parts), str(path)),
        )
        if matches:
            return matches[0]
    if required:
        raise FileNotFoundError(f"Không tìm thấy {label}: {sorted(accepted_names)}")
    return None


def resolve_contexts(override):
    if override is not None:
        path = Path(override)
        if not path.exists():
            raise FileNotFoundError(f"selected-contexts không tồn tại: {path}")
        return path

    for root in search_roots():
        candidates = sorted(
            root.rglob("*selected-contexts*"),
            key=lambda path: (len(path.parts), str(path)),
        )
        for path in candidates:
            if not path.is_dir():
                continue
            nested = path / "selected-contexts"
            if nested.is_dir() and any(nested.glob("context_*.json")):
                return nested
            if any(path.glob("context_*.json")) or any(path.rglob("context_*.json")):
                return path
        for path in candidates:
            if path.is_file() and path.suffix.casefold() == ".zip":
                return path
    return None


def model_snapshot_is_complete(model_dir):
    return (
        (model_dir / "config.json").is_file()
        and (
            any(model_dir.glob("*.safetensors"))
            or any(model_dir.glob("pytorch_model*.bin"))
        )
    )


def discover_generator_snapshot():
    if MODEL_PATH is not None:
        model_path = Path(MODEL_PATH)
        return str(model_path) if model_path.exists() else str(MODEL_PATH)
    for root in search_roots():
        for marker in root.rglob(MODEL_MARKER):
            try:
                payload = json.loads(marker.read_text(encoding="utf-8"))
            except (OSError, ValueError):
                continue
            if payload.get("repo_id") == GENERATOR_MODEL_ID and model_snapshot_is_complete(marker.parent):
                print(f"Tái sử dụng generator snapshot: {marker.parent}")
                return str(marker.parent)
    print("Không thấy snapshot; sẽ tải generator từ Hugging Face.")
    return GENERATOR_MODEL_ID


TRAIN_PATH = resolve_file("train set", TRAIN_PATH, {"train.json"})
PUBLIC_PATH = resolve_file(
    "public test",
    PUBLIC_PATH,
    {"public-official.json", "public-official(1).json", "public_official.json", "public_test.json"},
)
CONTEXTS_PATH = resolve_contexts(CONTEXTS_PATH)
SOURCE_SUBMISSION_PATH = resolve_file(
    "source submission",
    SOURCE_SUBMISSION_PATH,
    {"submission_v0.json"},
    required=False,
)
COMPARISON_SUBMISSION_PATH = resolve_file(
    "comparison submission",
    COMPARISON_SUBMISSION_PATH,
    {"submission_v1.json", "submission_v1(1).json", "submission_rag.json"},
    required=False,
)
GENERATOR_MODEL = discover_generator_snapshot()

print("train:", TRAIN_PATH)
print("public:", PUBLIC_PATH)
print("selected-contexts:", CONTEXTS_PATH or "không có — generation-only không cần retrieval")
print("source submission:", SOURCE_SUBMISSION_PATH or "không có — dùng đáp án train")
print("comparison submission:", COMPARISON_SUBMISSION_PATH or "không có")
print("generator:", GENERATOR_MODEL)


## 3. Chọn các đáp án dài để kiểm tra

Notebook ưu tiên `submission_v0.json` nếu tự tìm thấy hoặc được chỉ định qua `SOURCE_SUBMISSION_PATH`; câu hỏi tương ứng lấy từ public test. Nếu không có submission nguồn, notebook dùng câu hỏi/đáp án từ `train.json`. Khi có comparison submission, các ID bị rút ngắn mạnh được ưu tiên.


In [ ]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


train_data = load_json(TRAIN_PATH)
public_data = load_json(PUBLIC_PATH)
source_submission = load_json(SOURCE_SUBMISSION_PATH) if SOURCE_SUBMISSION_PATH else {}
comparison_data = load_json(COMPARISON_SUBMISSION_PATH) if COMPARISON_SUBMISSION_PATH else {}

assert isinstance(train_data, dict) and train_data, "train.json phải là JSON object không rỗng"
assert isinstance(public_data, dict) and public_data, "public test phải là JSON object không rỗng"

if source_submission:
    assert all(
        isinstance(value, dict) and isinstance(value.get("answer"), str)
        for value in source_submission.values()
    ), "source submission phải có dạng {id: {'answer': str}}"
    source_data = source_submission
    question_data = public_data
    source_label = str(SOURCE_SUBMISSION_PATH)
else:
    source_data = {
        str(qid): {"answer": str(item["answer"])}
        for qid, item in train_data.items()
        if isinstance(item, dict) and isinstance(item.get("answer"), str)
    }
    question_data = train_data
    source_label = str(TRAIN_PATH)

questions = {
    str(qid): str(item.get("question") or "")
    for qid, item in question_data.items()
    if isinstance(item, dict)
}
assert source_data, "Không có answer nguồn để chạy smoke test"

common_ids = set(source_data) & set(comparison_data) if comparison_data else set(source_data)
if comparison_data:
    ranked_ids = sorted(
        common_ids,
        key=lambda qid: len(source_data[qid]["answer"]) - len(comparison_data[qid].get("answer", "")),
        reverse=True,
    )
else:
    ranked_ids = sorted(common_ids, key=lambda qid: len(source_data[qid]["answer"]), reverse=True)

selected_ids = []
for qid in PRIORITY_IDS + ranked_ids:
    if qid in common_ids and qid not in selected_ids:
        selected_ids.append(qid)
    if len(selected_ids) >= TEST_LIMIT:
        break

selection_rows = []
for qid in selected_ids:
    source = source_data[qid]["answer"]
    comparison = comparison_data.get(qid, {}).get("answer", "")
    selection_rows.append({
        "id": qid,
        "question": questions.get(qid, ""),
        "source_chars": len(source),
        "comparison_chars": len(comparison) if comparison else None,
        "shrink_chars": len(source) - len(comparison) if comparison else None,
    })

selection_df = pd.DataFrame(selection_rows)
display(selection_df)
print(f"Nguồn context: {source_label}")
print(f"Đã chọn {len(selected_ids)} ID: {selected_ids}")


## 4. Load riêng Vi-Qwen2-1.5B-RAG

Cell này là bước nặng duy nhất. Không load embedding model hoặc reranker.


In [ ]:
set_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(
    GENERATOR_MODEL,
    trust_remote_code=False,
)

load_kwargs = {
    "trust_remote_code": False,
    "low_cpu_mem_usage": True,
}
if torch.cuda.is_available():
    load_kwargs.update({
        "device_map": "auto",
        "torch_dtype": torch.float16,
    })
else:
    load_kwargs["torch_dtype"] = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    GENERATOR_MODEL,
    **load_kwargs,
)
model.eval()

MODEL_DEVICE = next(model.parameters()).device
prompt_helper = ViQwenRAGGenerator(
    model_name_or_path=GENERATOR_MODEL,
    max_new_tokens=512,
    max_input_tokens=MAX_INPUT_TOKENS_512,
    repetition_penalty=REPETITION_PENALTY,
    seed=SEED,
)
prompt_helper._tokenizer = tokenizer
prompt_helper._model = model

print("Model device:", MODEL_DEVICE)
print("Model loaded:", GENERATOR_MODEL)


## 5. Prompt chống tóm tắt và hàm audit


In [ ]:
GENERIC_QUESTION = (
    "Hãy trả lại đầy đủ nội dung pháp lý cần thiết trong CONTEXT. "
    "Không tóm tắt, không lược bỏ danh sách, điều khoản hoặc biểu mẫu."
)


@torch.inference_mode()
def generate_one(question, context, max_input_tokens, max_new_tokens):
    model_context = int(
        getattr(model.config, "max_position_embeddings", max_input_tokens)
    )
    if max_new_tokens >= model_context:
        raise ValueError(
            f"max_new_tokens={max_new_tokens} phải nhỏ hơn context window={model_context}"
        )
    prompt_limit = min(max_input_tokens, model_context - max_new_tokens)
    inputs = prompt_helper._prepare_inputs(
        context=context,
        question=question or GENERIC_QUESTION,
        prompt_limit=prompt_limit,
    )
    inputs = {key: value.to(MODEL_DEVICE) for key, value in inputs.items()}
    input_length = inputs["input_ids"].shape[1]

    generate_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "repetition_penalty": REPETITION_PENALTY,
    }
    eos_token_id, pad_token_id = prompt_helper._generation_token_ids()
    if eos_token_id is not None:
        generate_kwargs["eos_token_id"] = eos_token_id
    if pad_token_id is not None:
        generate_kwargs["pad_token_id"] = pad_token_id

    started = time.time()
    outputs = model.generate(
        **inputs,
        **generate_kwargs,
    )
    elapsed = time.time() - started

    generated_ids = outputs[0, input_length:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return {
        "answer": answer,
        "input_tokens": int(input_length),
        "generated_tokens": int(generated_ids.shape[0]),
        "hit_token_limit": bool(generated_ids.shape[0] >= max_new_tokens - 4),
        "seconds": round(elapsed, 2),
    }


def word_tokens(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)


def source_coverage(source, prediction):
    src = Counter(word_tokens(source))
    pred = Counter(word_tokens(prediction))
    if not src:
        return 0.0
    return sum((src & pred).values()) / sum(src.values())


def audit(source, generated):
    answer = generated["answer"].strip()
    lower = answer.lower()
    source_words = len(word_tokens(source))
    answer_words = len(word_tokens(answer))
    possibly_cut = bool(
        len(answer) > 100
        and not answer.endswith((".", "!", "?", ")", "]", "}", '"', "”"))
    )
    result = {
        **generated,
        "source_words": source_words,
        "answer_words": answer_words,
        "length_ratio": round(answer_words / max(1, source_words), 3),
        "source_coverage": round(source_coverage(source, answer), 3),
        "has_boilerplate": bool("dựa trên ngữ cảnh" in lower or "theo ngữ cảnh" in lower),
        "says_no_information": bool("không đủ thông tin" in lower or "không có thông tin" in lower),
        "possibly_cut": possibly_cut,
    }
    result["passes_smoke_test"] = bool(
        not result["hit_token_limit"]
        and not result["has_boilerplate"]
        and not result["says_no_information"]
        and not result["possibly_cut"]
        and result["length_ratio"] >= 0.70
        and result["source_coverage"] >= 0.65
    )
    return result


## 6. Chạy smoke test 512 và 1024 token

Mỗi ID dùng đúng cùng một source context. Vì vậy khác biệt ở kết quả đến từ prompt/generation, không phải retrieval.


In [ ]:
configs = [
    ("strict_512", MAX_INPUT_TOKENS_512, 512),
]
if RUN_1024_TEST:
    configs.append(("strict_1024", MAX_INPUT_TOKENS_1024, 1024))

all_results = []
for item_index, qid in enumerate(selected_ids, start=1):
    context = source_data[qid]["answer"]
    question = (
        questions.get(qid) or GENERIC_QUESTION
        if USE_ACTUAL_QUESTIONS
        else GENERIC_QUESTION
    )
    print(f"\n[{item_index}/{len(selected_ids)}] ID={qid} | context={len(word_tokens(context))} words")

    for config_name, max_input_tokens, max_new_tokens in configs:
        generated = generate_one(
            question=question,
            context=context,
            max_input_tokens=max_input_tokens,
            max_new_tokens=max_new_tokens,
        )
        checked = audit(context, generated)
        record = {
            "id": qid,
            "config": config_name,
            "question": question,
            **checked,
        }
        all_results.append(record)
        print(
            f"  {config_name}: words={checked['answer_words']} "
            f"ratio={checked['length_ratio']:.3f} coverage={checked['source_coverage']:.3f} "
            f"tokens={checked['generated_tokens']} hit_limit={checked['hit_token_limit']} "
            f"pass={checked['passes_smoke_test']} ({checked['seconds']}s)"
        )

RESULT_JSON = EXPORT_DIR / "legalqa_generation_smoke_test_results.json"
RESULT_CSV = EXPORT_DIR / "legalqa_generation_smoke_test_summary.csv"

RESULT_JSON.write_text(
    json.dumps(all_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary_columns = [
    "id", "config", "source_words", "answer_words", "length_ratio",
    "source_coverage", "generated_tokens", "hit_token_limit",
    "has_boilerplate", "says_no_information", "possibly_cut",
    "passes_smoke_test", "seconds",
]
result_df = pd.DataFrame(all_results)
result_df[summary_columns].to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")

print("\nSaved:", RESULT_JSON)
print("Saved:", RESULT_CSV)


## 7. Kết luận nhanh

- `hit_token_limit=True`: ngân sách output chưa đủ; câu có khả năng bị cắt.
- `length_ratio < 0.70` hoặc `source_coverage < 0.65`: model vẫn rút quá mạnh với câu dài.
- Nếu 1024 vẫn thất bại, không nên bắt model sinh lại toàn bộ; submission nên trả raw chunk tốt nhất + chunk liền kề.


In [ ]:
summary_columns = [
    "id", "config", "source_words", "answer_words", "length_ratio",
    "source_coverage", "generated_tokens", "hit_token_limit",
    "has_boilerplate", "says_no_information", "possibly_cut",
    "passes_smoke_test", "seconds",
]

display(result_df[summary_columns].sort_values(["config", "passes_smoke_test", "length_ratio"]))

aggregate = result_df.groupby("config").agg(
    samples=("id", "count"),
    pass_rate=("passes_smoke_test", "mean"),
    mean_length_ratio=("length_ratio", "mean"),
    mean_source_coverage=("source_coverage", "mean"),
    hit_limit_rate=("hit_token_limit", "mean"),
    boilerplate_rate=("has_boilerplate", "mean"),
    no_information_rate=("says_no_information", "mean"),
    mean_seconds=("seconds", "mean"),
).reset_index()
display(aggregate)

best_config = aggregate.sort_values(
    ["pass_rate", "mean_source_coverage", "mean_length_ratio"],
    ascending=False,
).iloc[0]

print(f"Cấu hình tốt nhất trong smoke test: {best_config['config']}")
print(f"Pass rate: {best_config['pass_rate']:.1%}")

if best_config["pass_rate"] < 0.80:
    print("KẾT LUẬN: Generation vẫn không ổn định với đáp án dài. Hãy dùng raw context + adjacent chunks làm fallback khi submission.")
else:
    print("KẾT LUẬN: Prompt/token budget đã đủ tốt trên mẫu dài. Tiếp tục test thêm 20–50 câu trước submission.")

display(FileLink(str(RESULT_JSON)))
display(FileLink(str(RESULT_CSV)))


## 8. Xem chi tiết một vài câu


In [ ]:
for qid in selected_ids[:3]:
    display(Markdown(f"### ID {qid}"))
    question = (
        questions.get(qid) or GENERIC_QUESTION
        if USE_ACTUAL_QUESTIONS
        else GENERIC_QUESTION
    )
    print("QUESTION:", question)
    print("\nSOURCE CONTEXT:\n", source_data[qid]["answer"])
    for record in all_results:
        if record["id"] == qid:
            print(f"\n--- {record['config']} ---")
            print(record["answer"])
